In [1]:
import os
import cv2
import numpy as np

In [2]:
input_folder = r"C:\Users\TECH STORE\OneDrive\Desktop\IP_Project\enhanced_frames\match1"
output_folder = r"C:\Users\TECH STORE\OneDrive\Desktop\IP_Project\tracked_frames\match1"

os.makedirs(output_folder, exist_ok=True)

In [3]:
frame_files = sorted([
    f for f in os.listdir(input_folder)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

if not frame_files:
    raise ValueError("No frames found in input folder.")

print("Total frames:", len(frame_files))

Total frames: 164


In [4]:
first_frame_path = os.path.join(input_folder, frame_files[0])
first_frame = cv2.imread(first_frame_path)

if first_frame is None:
    raise ValueError("Could not read first frame.")

In [5]:
bbox = cv2.selectROI("Select Player", first_frame, fromCenter=False, showCrosshair=True)
cv2.destroyAllWindows()

if bbox == (0,0,0,0):
    raise ValueError("No player selected.")

In [6]:
if hasattr(cv2, "legacy") and hasattr(cv2.legacy, "TrackerCSRT_create"):
    tracker = cv2.legacy.TrackerCSRT_create()
elif hasattr(cv2, "TrackerCSRT_create"):
    tracker = cv2.TrackerCSRT_create()
else:
    raise AttributeError("CSRT tracker not available. Install opencv-contrib-python.")

tracker.init(first_frame, bbox)

print("Tracker initialized successfully.")

Tracker initialized successfully.


In [7]:
for file in frame_files:

    frame_path = os.path.join(input_folder, file)
    frame = cv2.imread(frame_path)

    if frame is None:
        print("Skipping unreadable frame:", file)
        continue

    success, box = tracker.update(frame)
    output = frame.copy()

    if success:
        x, y, w, h = [int(v) for v in box]

        # Draw bounding box
        cv2.rectangle(output, (x, y), (x+w, y+h), (255,255,255), 2)

    else:
        cv2.putText(output,
                    "Tracking Failure",
                    (20,40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255,255,255),
                    2)

    cv2.imwrite(os.path.join(output_folder, file), output)

print("Tracking completed.")
print("Tracked frames saved in:", output_folder)

Tracking completed.
Tracked frames saved in: C:\Users\TECH STORE\OneDrive\Desktop\IP_Project\tracked_frames\match1
